# 심층신경망(MLP) 딥러닝 파이프라인 (Deep MLP Pipeline via PyTorch)

- **Framework**: PyTorch (`torch.nn`, `torch.optim`)
- **Architecture**: Multi-Layer Perceptron (MLP) with Dropout & ReLU Activations
- **Objective**: 정형 데이터(Tabular) 환경에서 결정 트리(GBDT) 계열과 상이한 표현 학습(Representation Learning)을 수행하여 앙상블 다양성(Model Diversity) 확보

---

## 1. 모델 도입 배경 및 목표 (Rationale & Objectives)
1. **앙상블 다양성 확보 (Model Diversity for Ensembling)**:
   - 축 정렬 분할(Axis-aligned partitioning) 기반의 트리 모델과 달리, 연속적 잠재 공간(Continuous Latent Space)에서 비선형 결정 경계를 학습.
   - 후속 앙상블 파이프라인에서 단일 GBDT의 오분류를 보완하는 이종 아키텍처 컴포넌트 역할 수행.
2. **Tabular Deep Learning Baseline 표준화**:
   - 수치형 피처 표준화, 범주형 One-Hot 인코딩, 빈도 기반 피처 결합을 통한 신경망 전처리 규격 검증.
   - 수치적 안정성을 갖춘 이진 교차 엔트로피 손실(`BCEWithLogitsLoss`) 및 학습 최적화(AdamW + Early Stopping) 파이프라인 구현.

## 2. 네트워크 아키텍처 사양 (Network Architecture Specification)
```
Input Layer (D_in ≈ 33) 
  → Linear(D_in, 256) → ReLU() → Dropout(p=0.1) 
  → Linear(256, 128)   → ReLU() → Dropout(p=0.1) 
  → Linear(128, 1)     → Logit Output → Sigmoid → P(satisfaction=1)
```
- **Fully-Connected (Linear) Layers**: 가중치 행렬 $W$와 편향 $b$를 통한 선형 투영($z = Wx + b$).
- **Activation Function (ReLU)**: 비선형 활성화 함수($\max(0, x)$)를 적용하여 고차원 비선형 결정 경계 형성.
- **Regularization (Dropout)**: 학습 시 무작위 노드 비활성화($p=0.1$)를 통해 뉴런 간 상호 의존성(Co-adaptation) 억제 및 과적합 방지.

---

## 1. 환경 설정 및 데이터 로드 (Environment & Data Ingestion)

- **크로스 플랫폼 호환성**: Windows, macOS, Linux 환경에서 `pathlib.Path`를 통해 데이터셋 경로 자동 해결 및 디바이스(CUDA/CPU) 자동 매핑.
- **재현성 보장 (Reproducibility)**: Python 난수, NumPy, PyTorch CPU/CUDA 시드(`SEED = 42`)를 통합 고정.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
START_TIME = time.time()
np.random.seed(SEED)
torch.manual_seed(SEED)

# A GPU helps big networks; this small MLP trains fine on CPU (Kaggle: Settings -> Accelerator for a GPU).
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

# Chart style: thin marks, light grid, no top/right frame.
COLOR_MAIN = "#2a78d6"   # blue   - single-series charts, "satisfied"
COLOR_ALT = "#eb6834"    # orange - "not satisfied" / second series
COLOR_MUTED = "#8a8986"  # gray   - reference lines
plt.rcParams.update({
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "axes.titleweight": "bold",
})

In [ ]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)


def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))  # drop duplicates, keep order

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.\n"
            "- Kaggle: 오른쪽 Input 패널에서 대회 데이터를 추가하세요.\n"
            "- 로컬: 노트북과 같은 위치의 data/ 폴더에 파일을 넣으세요."
        )
    raise RuntimeError(
        "데이터 폴더 후보가 여러 개입니다. 사용할 폴더를 DATA_DIR에 직접 지정하세요:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


DATA_DIR = find_data_dir()
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "satisfaction"
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
NUM_COLS = [c for c in train.columns if c not in CAT_COLS + ["id", TARGET]]
RATING_COLS = [c for c in NUM_COLS if train[c].max() <= 5]
y = train[TARGET].astype(int).values

print("DATA_DIR =", DATA_DIR)
print("train", train.shape, "| test", test.shape)

In [ ]:
# Where this notebook writes its predictions (Kaggle only allows writing under /kaggle/working).
OUT_DIR = Path("/kaggle/working/outputs") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR =", OUT_DIR.resolve())


N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(train, y))

# fold_id[i] = the fold in which row i is used for validation (same in every notebook of the series).
fold_id = np.full(len(train), -1)
for k, (_, va_idx) in enumerate(FOLDS):
    fold_id[va_idx] = k
assert (fold_id >= 0).all()


def save_predictions(name: str, oof: np.ndarray, test_pred: np.ndarray) -> None:
    """Save OOF and test predictions keyed by id, so notebook 07 can align them safely."""
    assert np.isfinite(oof).all() and np.isfinite(test_pred).all()
    pd.DataFrame({"id": train["id"].values, "fold": fold_id, "pred": oof}).to_csv(OUT_DIR / f"oof_{name}.csv", index=False)
    pd.DataFrame({"id": test["id"].values, "pred": test_pred}).to_csv(OUT_DIR / f"test_{name}.csv", index=False)
    print(f"saved oof_{name}.csv / test_{name}.csv")


FREQ_COLS = ["Age", "Flight Distance"]


def add_freq_features(fit_df: pd.DataFrame, *apply_dfs: pd.DataFrame) -> list:
    """Frequency encoding fit on fit_df only (no peeking at validation/test), applied to every frame."""
    out = []
    for df in (fit_df, *apply_dfs):
        df = df.copy()
        for col in FREQ_COLS:
            counts = fit_df[col].value_counts()
            df[f"{col}_freq"] = df[col].map(counts).fillna(0).astype(float)
        out.append(df)
    return out

In [ ]:
X = train.drop(columns=["id", TARGET])
X_test = test.drop(columns=["id"])
for col in CAT_COLS:
    categories = sorted(train[col].unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)

## 2. 데이터 전처리 파이프라인 (Data Preprocessing Pipeline)

경사하강법(Gradient Descent) 기반 최적화의 수렴 안정성을 위한 데이터 정규화 및 인코딩 정의:
- **수치형 피처 (Numerical Features)**: 
  - `SimpleImputer(strategy='median')`: 결측치 대체.
  - `StandardScaler`: $z$-score 정규화($\mu=0, \sigma=1$)를 통해 피처 간 스케일 불균형(`Flight Distance` 등)으로 인한 기울기 편향 방지.
- **범주형 피처 (Categorical Features)**: `OneHotEncoder(handle_unknown='ignore')`를 통한 이진 희소 벡터화.
- **파생 피처 (Feature Engineering)**: 범주형 변수의 발생 빈도를 정량화한 빈도 인코딩(Frequency Encoding) 피처 결합.
- **데이터 누수 방지 (Leakage Prevention)**: 스케일러 및 통계량 추정은 각 Fold의 학습 세트(`X_tr`)에서만 Fit 수행.

In [ ]:
def prepare_fold(tr_idx, va_idx):
    X_tr, X_va, X_te = add_freq_features(X.iloc[tr_idx], X.iloc[va_idx], X_test)
    num_cols = [c for c in X_tr.columns if c not in CAT_COLS]
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
    ])
    arrays = [prep.fit_transform(X_tr)] + [prep.transform(df) for df in (X_va, X_te)]
    return [torch.tensor(a, dtype=torch.float32) for a in arrays]


Xt_tr, Xt_va, Xt_te = prepare_fold(*FOLDS[0])
print("input features after preprocessing:", Xt_tr.shape[1])
print("first fold tensors:", tuple(Xt_tr.shape), tuple(Xt_va.shape), tuple(Xt_te.shape))

## 3. 모델 아키텍처 구현 (Model Architecture Implementation)

PyTorch `nn.Module` 기반의 Multi-Layer Perceptron (MLP) 클래스 정의.

- **출력단 (Logit Output)**: 네트워크 최종 출력은 Sigmoid 적용 전의 Raw Logit을 반환.
- **손실 함수 (Loss Function)**: `nn.BCEWithLogitsLoss`를 채택하여 Log-Sum-Exp 트릭을 통한 수치적 안정성(Numerical Stability) 확보 및 언더플로우/오버플로우 방지.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_inputs: int, hidden=(256, 128), dropout: float = 0.1):
        super().__init__()
        layers, prev = [], n_inputs
        for size in hidden:
            layers += [nn.Linear(prev, size), nn.ReLU(), nn.Dropout(dropout)]
            prev = size
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)  # logits


print(MLP(Xt_tr.shape[1]))
print("trainable parameters:", sum(p.numel() for p in MLP(Xt_tr.shape[1]).parameters()))

## 4. 학습 및 검증 루프 설계 (Training & Validation Engine)

- **미니배치 최적화 (Mini-batch Optimization)**: `BATCH_SIZE = 2048` 설정을 통해 대규모 정형 데이터셋($N \approx 560,000$)에 대한 계산 효율성 및 안정적 기울기 추정.
- **옵티마이저 (Optimizer)**: `AdamW (weight_decay=1e-4, lr=1e-3)`를 사용하여 적응형 학습률 적용 및 $L_2$ 가중치 감쇠(Weight Decay) 규제.
- **검증 평가 (Validation Tracking)**: 매 Epoch 종료 시 Validation 세트에 대한 ROC-AUC를 산출하여 모델 수렴 추적.
- **조기 종료 및 체크포인트 복원 (Early Stopping & Model Checkpointing)**: 
  - `PATIENCE = 4` 에폭 동안 검증 AUC 개선 부재 시 조기 종료.
  - 최적 검증 AUC를 기록한 에폭의 모델 파라미터(`best_state`)를 최종 추론에 반영.
- **동작 모드 전환 (Evaluation Mode)**: `model.train()`과 `model.eval()`을 명시적으로 전환하여 추론 시 Dropout 비활성화 및 Deterministic forward 보장.

In [ ]:
BATCH_SIZE = 2048
MAX_EPOCHS = 40
PATIENCE = 4
LR = 2e-3


@torch.no_grad()
def predict(model, X_tensor):
    model.eval()
    out = [torch.sigmoid(model(X_tensor[i:i + 65_536].to(DEVICE))).cpu() for i in range(0, len(X_tensor), 65_536)]
    return torch.cat(out).numpy()


def train_fold(Xt_tr, y_tr, Xt_va, y_va, Xt_te):
    torch.manual_seed(SEED)
    model = MLP(Xt_tr.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()
    yt_tr = torch.tensor(y_tr, dtype=torch.float32)

    best = {"auc": -1.0, "epoch": 0, "va_pred": None, "te_pred": None}
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        order = torch.randperm(len(Xt_tr))
        total_loss = 0.0
        for start in range(0, len(order), BATCH_SIZE):
            idx = order[start:start + BATCH_SIZE]
            xb, yb = Xt_tr[idx].to(DEVICE), yt_tr[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)

        va_pred = predict(model, Xt_va)
        va_auc = roc_auc_score(y_va, va_pred)
        history.append({"epoch": epoch, "train_loss": total_loss / len(order), "valid_auc": va_auc})
        if va_auc > best["auc"]:
            best = {"auc": va_auc, "epoch": epoch, "va_pred": va_pred, "te_pred": predict(model, Xt_te)}
        elif epoch - best["epoch"] >= PATIENCE:
            break
    return best, pd.DataFrame(history)

## 5. Stratified 5-Fold 교차 검증 및 추론 (Cross-Validation Execution)

In [ ]:
t0 = time.time()
oof, test_pred, fold_aucs, histories = np.full(len(X), np.nan), np.zeros(len(X_test)), [], []
for k, (tr_idx, va_idx) in enumerate(FOLDS):
    Xt_tr, Xt_va, Xt_te = prepare_fold(tr_idx, va_idx)
    best, history = train_fold(Xt_tr, y[tr_idx], Xt_va, y[va_idx], Xt_te)
    oof[va_idx] = best["va_pred"]
    test_pred += best["te_pred"] / N_FOLDS
    fold_aucs.append(best["auc"])
    histories.append(history)
    print(f"fold {k + 1}: AUC {best['auc']:.5f} at epoch {best['epoch']} (stopped at {len(history)}) | {time.time() - t0:.0f}s")

mlp_auc = roc_auc_score(y, oof)
print(f"\nMLP OOF AUC = {mlp_auc:.5f}")
save_predictions("mlp", oof, test_pred)

In [ ]:
from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for k in list(range(1, len(histories))) + [0]:  # draw fold 1 last so it stays on top
    h = histories[k]
    style = {"color": COLOR_MAIN if k == 0 else COLOR_MUTED, "linewidth": 2 if k == 0 else 1,
             "label": "fold 1" if k == 0 else ("other folds" if k == 1 else None)}
    axes[0].plot(h["epoch"], h["train_loss"], **style)
    axes[1].plot(h["epoch"], h["valid_auc"], **style)
    axes[1].scatter(h.loc[h["valid_auc"].idxmax(), "epoch"], h["valid_auc"].max(), s=25, color=style["color"], zorder=3)
axes[0].set_title("Training loss (lower is better)")
axes[1].set_title("Validation AUC (higher is better, dot = best epoch)")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### 📌 수렴 분석 및 모델 성능 검증 (Convergence & Performance Analysis)

- **학습 곡선 분석 (Learning Curve Dynamics)**:
  - Training Loss는 지속적으로 감소하나, Validation AUC는 $10 \sim 14$ Epoch 부근에서 포화(Plateau)에 도달하여 일반화 한계점 형성.
  - Early Stopping 메커니즘을 통해 최적 일반화 성능 시점의 체크포인트 가중치를 안정적으로 보존.
- **정량적 성능 지표 (Quantitative Comparison)**:
  - **MLP OOF ROC-AUC**: **0.9573** (kNN Baseline 0.946 대비 대폭 우세, 단일 GBDT 0.9593 대비 $-0.0020$ 차이로 경쟁력 있는 성능 확보).
- **Fold 간 변동성**: 가중치 초기화 난수성 및 배치 셔플링의 영향으로 Fold별 최적 수렴 Epoch($10 \sim 14$) 및 세부 점수에 소폭의 분산 존재.

---

### 🚀 향후 아키텍처 고도화 과제 (Engineering Roadmap)

| 개선 영역 | 세부 구현 전략 |
|---|---|
| **임베딩 레이어 도입 (Entity Embeddings)** | 고차원 범주형 변수를 원-핫 대신 저차원 Dense Embedding 공간으로 매핑 |
| **순서형 피처 임베딩 (Ordinal Embedding)** | 13개 서비스 평점($0 \sim 5$)을 카테고리로 간주하고 개별 임베딩 레이어로 표현 |
| **하이퍼파라미터 최적화** | Optuna 기반 Layer Depth, Hidden Units, Dropout Rate($p$), Weight Decay 튜닝 |
| **Multi-Seed Bagging** | 상이한 Seed로 학습된 $N$개 MLP 모델의 앙상블을 통한 가중치 초기화 분산 감소 |

In [ ]:
# print(f"전체 실행 시간: {time.time() - START_TIME:.0f}초")